# DRL Training

This notebook configures, trains, saves, and evaluates a DQN agent for the coupled climate-social system environment.


In [ ]:
import datetime
import math
import os
import random
import sys

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io
import torch
from gymnasium import spaces
from matplotlib.gridspec import GridSpec
from scipy.integrate import odeint
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.logger import configure
from stable_baselines3.common.monitor import Monitor

# Add the project source directory to the Python module search path.
sys.path.append(os.path.abspath("src"))

from src.envs.iseec_lx_v5_pomdp_without_masking_all_actions import IEMEnv


def set_global_seed(seed):
    """Set random seeds for reproducible experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# Optional: uncomment the following lines to use a fixed global seed.
# SEED = 42
# set_global_seed(SEED)

# Centralized experiment configuration.
base_config = {
    # MDP settings
    "custom_reward_type": "weight_three_obj_over_Ta",
    # Environment and policy
    "env_id": "iseec_lx_v5_ste_without_masking",
    "rl_model_name": "DQN",
    "policy": "MlpPolicy",
    "network_name": "Netxxx_no_big_env",
    # Training duration
    "total_timesteps_diy": 900000,
    # Example:
    # policy_kwargs = dict(
    #     activation_fn=th.nn.ReLU,
    #     net_arch=[256, 256, dict(pi=[128, 64], vf=[128, 64])],
    # )
    "policy_kwargs_diy": "dict_pi_vf_default",
    # Optimizer and network settings
    "hyperparams_diy": "default_seed100",
    # "learning_rate": 3e-4,
    # "ent_coef": 0.0,
    # "batch_size": 64,
    # "n_epochs": 10,
    # Replay and update settings (PPO examples)
    # "gamma": 0.99,
    # "clip_range": 0.2,
    # "gae_lambda": 0.95,
    # Randomization settings
    "use_random_reset": False,
    "seed": 100,
    "po_mdp_state": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
}

# Build a descriptive name for logs and model artifacts.
log_name = (
    f'{base_config["env_id"]}_'
    f'{base_config["rl_model_name"]}_'
    f'{base_config["custom_reward_type"]}_'
    f'{int(base_config["total_timesteps_diy"])}_'
    f'{base_config["hyperparams_diy"]}'
)
os.makedirs(f"./logs/{log_name}", exist_ok=True)

# Expose frequently used configuration values.
custom_reward_type = base_config["custom_reward_type"]
network_name = base_config["network_name"]
rl_model_name = base_config["rl_model_name"]
total_timesteps_diy = base_config["total_timesteps_diy"]
policy_kwargs_diy = base_config["policy_kwargs_diy"]

# Configure CSV, TensorBoard, and JSON logging.
new_logger = configure(
    f"logs/{log_name}",
    [
        "csv",
        "tensorboard",
        "json",
    ],
)

# Initialize the environment.
env = IEMEnv(
    reward_type=base_config["custom_reward_type"],
    seed=base_config["seed"],
    pomdp_state_indices=base_config["po_mdp_state"],
)

env.reward_weights = {
    "reward_weight_three_obj_over_Ta": {
        "Ta": 5,
        # "Ca": 10,
        # "energy": 15,
        "over": 20,
    },
}

env_monitor = Monitor(env, f"./logs/{log_name}")

# Alternative PPO configuration retained for reference.
# model = PPO(
#     "MlpPolicy",
#     env_monitor,
#     # ent_coef=0.01,
#     verbose=1,
# )

model = DQN(
    "MlpPolicy",
    env_monitor,
    verbose=1,
)

model.set_logger(new_logger)


## Environment Check

Inspect representative samples from the action and observation spaces before training.


In [ ]:
# Uncomment to run the Stable-Baselines3 environment checker.
# check_env(env_monitor)

print(log_name)
print("Action space sample:", env.action_space.sample())
print("Observation space sample:", env.observation_space.sample())
# print(model.policy)


## Training Curves


In [ ]:
def plot_callback_reward(metrics):
    """Plot total and dimension-level episode rewards collected during training."""
    plt.figure(figsize=(20, 16))

    episodes = range(len(metrics["rewards"]))

    # Plot the total episode reward and its moving statistics.
    plt.subplot(321)
    plt.title(
        f'Step: {metrics["step_idx"]}, '
        f'Latest reward: {metrics["rewards"][-1]:.2f}\n'
        f'Episode: {metrics["episodes"]}, '
        f'Moving avg: {metrics["moving_avg_rewards"][-1]:.2f}'
    )
    plt.plot(
        episodes,
        metrics["rewards"],
        label="Raw rewards",
        color="gray",
        alpha=0.3,
    )
    plt.plot(
        episodes,
        metrics["moving_avg_rewards"],
        label="Moving average",
        color="blue",
        linewidth=2,
    )

    moving_avg = np.array(metrics["moving_avg_rewards"])
    moving_std = np.array(metrics["moving_std_rewards"])
    plt.fill_between(
        episodes,
        moving_avg - 0.25 * moving_std,
        moving_avg + 0.25 * moving_std,
        alpha=0.1,
    )
    plt.fill_between(
        episodes,
        moving_avg - 0.5 * moving_std,
        moving_avg + 0.5 * moving_std,
        alpha=0.1,
    )
    plt.xlabel("Episodes")
    plt.ylabel("Total Episode Reward")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Plot rewards for each objective dimension.
    reward_dims = [
        "reward_dim1",
        "reward_dim2",
        "reward_dim3",
        "reward_dim4",
        "reward_dim5",
    ]
    titles = [
        "Reward Dim 1",
        "Reward Dim 2",
        "Reward Dim 3",
        "Reward Dim 4",
        "Reward Dim 5",
    ]
    colors = ["green", "orange", "purple", "red", "blue"]

    for index, (dimension, title, color) in enumerate(
        zip(reward_dims, titles, colors)
    ):
        plt.subplot(3, 2, index + 2)
        plt.title(title)
        plt.plot(episodes, metrics[dimension], label=title, color=color)
        plt.xlabel("Episodes")
        plt.ylabel(title)
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


class TrainingMonitorCallback(BaseCallback):
    """Collect episode-level total and dimension-specific rewards."""

    def __init__(self, verbose=1, window_size=50):
        super().__init__(verbose)
        self.window_size = window_size
        self.data = {
            "rewards": [],
            "moving_avg_rewards": [],
            "moving_std_rewards": [],
            "reward_dim1": [],
            "reward_dim2": [],
            "reward_dim3": [],
            "reward_dim4": [],
            "reward_dim5": [],
            "episodes": 0,
            "step_idx": 0,
        }
        self.episode_rewards = 0
        self.episode_dim1 = 0
        self.episode_dim2 = 0
        self.episode_dim3 = 0
        self.episode_dim4 = 0
        self.episode_dim5 = 0

    def _on_step(self):
        """Update reward statistics after each environment step."""
        reward = self.locals.get("rewards")[0]
        info = self.locals.get("infos")[0]

        self.episode_rewards += reward
        self.data["step_idx"] += 1

        self.episode_dim1 += info["reward"]["dim1"]
        self.episode_dim2 += info["reward"]["dim2"]
        self.episode_dim3 += info["reward"]["dim3"]
        self.episode_dim4 += info["reward"].get("dim4", 0)
        self.episode_dim5 += info["reward"].get("dim5", 0)

        if self.locals.get("dones")[0]:
            self.data["rewards"].append(self.episode_rewards)

            recent_rewards = self.data["rewards"][-self.window_size :]
            moving_avg = np.mean(recent_rewards)
            moving_std = (
                np.std(recent_rewards) if len(recent_rewards) > 1 else 0
            )
            self.data["moving_avg_rewards"].append(moving_avg)
            self.data["moving_std_rewards"].append(moving_std)

            self.data["reward_dim1"].append(self.episode_dim1)
            self.data["reward_dim2"].append(self.episode_dim2)
            self.data["reward_dim3"].append(self.episode_dim3)
            self.data["reward_dim4"].append(self.episode_dim4)
            self.data["reward_dim5"].append(self.episode_dim5)

            # Reset episode-level accumulators.
            self.data["episodes"] += 1
            self.episode_rewards = 0
            self.episode_dim1 = 0
            self.episode_dim2 = 0
            self.episode_dim3 = 0
            self.episode_dim4 = 0
            self.episode_dim5 = 0

        return True

    def get_metrics(self):
        """Return all collected training metrics."""
        return self.data


# Create the training callback with a 50-episode moving window.
callback = TrainingMonitorCallback(window_size=50)

# Train the model.
model.learn(
    total_timesteps=int(total_timesteps_diy),
    callback=callback,
    log_interval=1,
)

# Retrieve and visualize the collected metrics.
metrics = callback.get_metrics()
plot_callback_reward(metrics)


In [ ]:
# Save the trained model with a timestamped filename.
current_time = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"当前时间: {current_time}")

model.save(f"./model/{log_name}_{current_time}")
print(f"模型已经保存在 ./model/{log_name}_{current_time}")


# Model Evaluation


## Load the Trained Model


In [ ]:
# Alternative loading examples retained for reference.
# model = PPO.load(f"./model/{log_name}", env=env_monitor)
# model = DQN.load(f"./model/{log_name}", env=env_monitor)

model = DQN.load(
    f"./model/{log_name}_{current_time}",
    env=env_monitor,
)


## Visualize Evaluation Results


In [ ]:
# Run one deterministic evaluation episode and render the final state.
episodes = 1
max_steps = 250

for episode_index in range(episodes):
    obs, _ = env_monitor.reset(
        use_random_reset=base_config["use_random_reset"],
        seed=base_config["seed"],
    )

    episode_reward = 0
    done = False

    for step_index in range(max_steps):
        # To test random actions instead:
        # action = env_monitor.action_space.sample()
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, _, info = env_monitor.step(action)

        # Optional debugging output:
        # print(
        #     f"year {env_monitor.t} - Step {step_index} - "
        #     f"Action: {action} - Reward: {reward} - Done: {done}"
        # )
        # print(f"Info: {info['state_values']['T_a']}")

        episode_reward += reward

        if done:
            env_monitor.render()
            plt.show()
            plt.pause(0.1)
            break

    print(f"Episode {episode_index} finished with reward {reward}")
    print(f"Final T_a: {obs[0]}")
    print(f"Final C_a: {obs[1]}")
    print(
        "Final energy: "
        f"{env_monitor.energy_MYadjusted18502100_total_plus_B3B_plus_ACE3[-1]}"
    )
